In [ ]:
!pip install pennylane

In [ ]:
import os
import torch
import pennylane as qml
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!cp -r /content/drive/MyDrive/dataset /content/

cp: cannot create regular file '/content/dataset/train/yellow/yellow leaf disease_original_960.jpg_acefca22-e9d6-4c53-bd5e-7a0eb3de4e37.jpg': File exists
cp: cannot create regular file '/content/dataset/train/yellow/yellow leaf disease_original_963.jpg_1e56c828-b353-470d-b78e-e3c538b263cd.jpg': File exists
cp: cannot create regular file '/content/dataset/train/yellow/yellow leaf disease_original_964.jpg_0afb5985-01c3-4279-ba67-4883476f315f.jpg': File exists
cp: cannot create regular file '/content/dataset/train/yellow/yellow leaf disease_original_963.jpg_5e9546ba-ea52-4ff4-8284-d75effa95479.jpg': File exists
cp: cannot create regular file '/content/dataset/train/yellow/yellow leaf disease_original_963.jpg_46358e88-3f69-40ca-809d-8d3044e61ada.jpg': File exists
cp: cannot create regular file '/content/dataset/train/yellow/yellow leaf disease_original_967.jpg_22a1fa1a-4fbd-410d-838a-7b15c209e46a.jpg': File exists
cp: cannot create regular file '/content/dataset/train/yellow/yellow leaf di

In [ ]:
train_dir = "/content/dataset/train"
test_dir = "/content/dataset/test"

In [ ]:
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

In [ ]:
train_dataset = torchvision.datasets.ImageFolder(
    train_dir,
    transform=transform
)

test_dataset = torchvision.datasets.ImageFolder(
    test_dir,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False
)

In [ ]:
n_qubits = 4

dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def quantum_circuit(inputs, weights):

    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)

    qml.templates.StronglyEntanglingLayers(
        weights,
        wires=range(n_qubits)
    )

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

In [ ]:
weight_shapes = {"weights": (3, n_qubits, 3)}

qnn = qml.qnn.TorchLayer(
    quantum_circuit,
    weight_shapes
)

In [ ]:
class HybridModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3,16,3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16,32,3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1,1))
        )

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(32,4)

        self.qnn = qnn

        self.fc2 = nn.Linear(4,2)

    def forward(self,x):

        x = self.conv(x)
        x = self.flatten(x)
        x = self.fc1(x)

        x = x.float()

        # QNN batch workaround
        qnn_outputs = []

        for i in range(x.shape[0]):
            q_out = self.qnn(x[i])
            qnn_outputs.append(q_out)

        x = torch.stack(qnn_outputs)

        x = self.fc2(x)

        return x

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = HybridModel().to(device)

In [ ]:
healthy = 605
yellow = 1477

class_weights = torch.tensor([
    yellow/healthy,
    1.0
]).to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
def calculate_accuracy(loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    return 100 * correct / total

In [ ]:
checkpoint_path = "/content/drive/MyDrive/arecanut_qnn.pth"

start_epoch = 0
best_acc = 0

if os.path.exists(checkpoint_path):

    print("Loading checkpoint...")

    checkpoint = torch.load(checkpoint_path)

    model.load_state_dict(checkpoint['model'])

    optimizer.load_state_dict(checkpoint['optimizer'])

    start_epoch = checkpoint['epoch']
    best_acc = checkpoint.get('best_acc', 0)

In [ ]:
epochs = 40

for epoch in range(start_epoch, epochs):

    model.train()

    total_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()


    train_acc = calculate_accuracy(train_loader)
    test_acc = calculate_accuracy(test_loader)

    if test_acc > best_acc:
      best_acc = test_acc
      torch.save(model.state_dict(),
                "/content/drive/MyDrive/best_arecanut_qnn.pth")

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {total_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Test Accuracy: {test_acc:.2f}%")

    # Save checkpoint
    torch.save({
        'epoch': epoch+1,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'best_acc': best_acc
    }, checkpoint_path)

    print("Checkpoint Saved")


Epoch 1
Loss: 292.4044
Train Accuracy: 72.72%
Test Accuracy: 72.80%
Checkpoint Saved

Epoch 2
Loss: 264.0501
Train Accuracy: 77.47%
Test Accuracy: 77.39%
Checkpoint Saved

Epoch 3
Loss: 236.4817
Train Accuracy: 85.98%
Test Accuracy: 87.74%
Checkpoint Saved

Epoch 4
Loss: 198.3430
Train Accuracy: 87.03%
Test Accuracy: 88.89%
Checkpoint Saved

Epoch 5
Loss: 165.4159
Train Accuracy: 87.80%
Test Accuracy: 87.16%
Checkpoint Saved

Epoch 6
Loss: 149.0304
Train Accuracy: 89.24%
Test Accuracy: 89.27%
Checkpoint Saved

Epoch 7
Loss: 139.7668
Train Accuracy: 91.50%
Test Accuracy: 92.34%
Checkpoint Saved

Epoch 8
Loss: 128.8267
Train Accuracy: 93.13%
Test Accuracy: 93.68%
Checkpoint Saved

Epoch 9
Loss: 111.8462
Train Accuracy: 80.60%
Test Accuracy: 81.99%
Checkpoint Saved

Epoch 10
Loss: 114.7105
Train Accuracy: 93.37%
Test Accuracy: 94.44%
Checkpoint Saved

Epoch 11
Loss: 95.9046
Train Accuracy: 95.77%
Test Accuracy: 95.59%
Checkpoint Saved

Epoch 12
Loss: 98.1286
Train Accuracy: 93.71%
Test A

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

print("Accuracy:", 100*correct/total)